In [1]:
# ============================================================
# FUNCTION 4 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

# Reduce thread pressure in the final-week notebooks
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function4/initial_inputs.npy"
)

Y = np.load(
    "function4/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------

week12_x = np.array([
    0.361985,
    0.401083,
    0.421437,
    0.427725
])

week12_actual = (
    0.627826619411398
)

# Week 12 x2-axis profile prediction
week12_pred_mean = (
    0.4810118045924785
)

week12_pred_std = (
    0.23983028665032957
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week12_pred_mean
)

print(
    "Predicted std :",
    week12_pred_std
)

print(
    "Actual        :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Recent local evidence
# ------------------------------------------------------------

week10_x = np.array([
    0.366845,
    0.409623,
    0.430833,
    0.419658
])

week10_y = (
    0.6208031832357475
)

week11_x = np.array([
    0.371677,
    0.403797,
    0.426584,
    0.419342
])

week11_y = (
    0.6140414365565303
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nIncumbent:")
print(
    best_x,
    "->",
    best_y
)

print("\nWeek 10:")
print(
    week10_x,
    "->",
    week10_y
)

print("\nWeek 11:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_actual
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Final week:
# remain tightly centred on the proven basin.

empirical_cap = min(
    1.5 * nearest_distance,
    0.025
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed final-week trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.12 * lengthscales,
    0.008,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        350000,
        4
    )
)

# Match actual portal precision
candidates = np.round(
    candidates,
    6
)

distance, _ = tree.query(
    candidates,
    k=1
)

# F4 has consistently used the 0.008 duplicate threshold
candidates = candidates[
    distance > 0.008
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 14. Axis-profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

axis_best = {}

for j in range(4):

    values = np.linspace(
        lower[j],
        upper[j],
        40000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.008
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    axis_best[j] = profile_valid[idx]

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 15. Week 12 mirror diagnostic
# ------------------------------------------------------------
#
# Week 12 decreased x2 from:
#
# 0.412010 -> 0.401083
#
# and performance fell.
#
# Mirror that exact displacement to the OTHER side of the
# incumbent while holding x1, x3 and x4 fixed.
# ------------------------------------------------------------

week12_step = (
    week12_x
    - best_x
)

mirror_candidate = (
    best_x
    - week12_step
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, mirror_idx = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "Mirror candidate =",
    mirror_candidate
)

print(
    "Nearest observed distance =",
    mirror_dist[0]
)

print(
    "Nearest observed point =",
    X[mirror_idx[0]]
)

if mirror_dist[0] > 0.008:

    mirror_mu, mirror_sigma = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        mirror_mu[0]
    )

    print(
        "std =",
        mirror_sigma[0]
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )


# ------------------------------------------------------------
# 16. Incumbent-centred x2 profile
# ------------------------------------------------------------
#
# x2 is especially interesting because Week 12 changed only
# this coordinate.
# ------------------------------------------------------------

x2_values = np.linspace(
    lower[1],
    upper[1],
    60000
)

x2_profile = np.tile(
    best_x,
    (
        len(x2_values),
        1
    )
)

x2_profile[:, 1] = x2_values

x2_profile = np.round(
    x2_profile,
    6
)

x2_distance, _ = tree.query(
    x2_profile,
    k=1
)

x2_valid = x2_profile[
    x2_distance > 0.008
]

print("\n================================")
print("FINAL-WEEK x2 PROFILE")
print("================================")

if len(x2_valid) > 0:

    x2_mu, x2_sigma = gp.predict(
        x2_valid,
        return_std=True
    )

    x2_idx = np.argmax(
        x2_mu
    )

    print(
        "Best x2 candidate =",
        x2_valid[x2_idx]
    )

    print(
        "mean =",
        x2_mu[x2_idx]
    )

    print(
        "std =",
        x2_sigma[x2_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            x2_valid[x2_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible x2-profile candidates."
    )

DATA
X shape: (42, 4)
Y shape: (42,)

Current best:
[0.361985 0.41201  0.421437 0.427725] -> 0.6704983808455585

Y range:
min = -32.625660215962455
max = 0.6704983808455585
std = 9.88052648805564

WEEK 12 CALIBRATION CHECK
Predicted mean: 0.4810118045924785
Predicted std : 0.23983028665032957
Actual        : 0.627826619411398

Prediction error: 0.14681481481891945

Error / predicted std: 0.6121612781665651

RECENT LOCAL EVIDENCE

Incumbent:
[0.361985 0.41201  0.421437 0.427725] -> 0.6704983808455585

Week 10:
[0.366845 0.409623 0.430833 0.419658] -> 0.6208031832357475

Week 11:
[0.371677 0.403797 0.426584 0.419342] -> 0.6140414365565303

Week 12:
[0.361985 0.401083 0.421437 0.427725] -> 0.627826619411398

GP FIT

Fitted kernel:
2.66**2 * Matern(length_scale=[1.66, 1.51, 1.4, 1.49], nu=2.5) + WhiteKernel(noise_level=0.000439)

ARD lengthscales:
[1.65817024 1.50945341 1.39809312 1.4939981 ]

Normalised inverse-lengthscale sensitivity:
[0.22756058 0.24998068 0.26989203 0.25256671]

EMPIRI

In [2]:
# ============================================================
# FINAL FUNCTION 4 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - The incumbent remains the strongest observed point.
# - Week 12 isolated a downward x2 move and performance fell.
# - Week 10 and Week 11 also involved lower x2 values and did
#   not beat the incumbent.
# - Although the unrestricted GP mean/UCB still prefers lower
#   x2, the real observations argue against repeating that move.
#
# Among the one-coordinate profiles, x4 gives the strongest
# posterior mean while keeping x1, x2 and x3 exactly at the
# proven incumbent values.
#
# Therefore use a conservative x4-only final refinement.
# ============================================================

week13_candidate = np.array([
    0.361985,
    0.412010,
    0.421437,
    0.416700
])

print("================================")
print("FUNCTION 4 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Distance from current incumbent
# ------------------------------------------------------------

distance_from_best = np.linalg.norm(
    week13_candidate - best_x
)

print("\nDistance from current best:")
print(distance_from_best)


# ------------------------------------------------------------
# Check against all previous observations
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.008
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 4 - WEEK 13 FINAL

Candidate:
[0.361985 0.41201  0.421437 0.4167  ]

Predicted mean:
0.5026244476667632

Predicted std:
0.22120311328815626

Distance from current best:
0.011025000000000007

Nearest observed-point distance:
0.011025000000000007

Nearest observed point:
[0.361985 0.41201  0.421437 0.427725]

Duplicate rule satisfied:
True

Portal format:
0.361985-0.412010-0.421437-0.416700
